In [1]:
import json

In [2]:
import pickle

In [3]:
import tqdm

In [4]:
from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [6]:
import pandas as pd

In [7]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [8]:
MODEL_ID = "google/gemma-4-E2B-it"

In [9]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [10]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [11]:
df = pd.read_excel("data_clean.xlsx", header=0)

In [12]:
df.head()

,title_rus,title_eng,annotation,description
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про..."
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,..."
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...",Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют..."
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...


In [13]:
df.tail()

,title_rus,title_eng,annotation,description
1182,Влияние мер контроля за движением капитала на ...,The Impact of Capital Controls on the Investme...,NaN,NaN
1183,"Создание страницы (лендинга) проектов ПУЛ ""Раз...",Creation of a Landing Page for Projects of the...,Проект направлен на разработку и запуск одност...,Проектно-учебная лаборатория «Развитие универс...
1184,Разработка стратегической и контентно-коммуник...,Development of a Strategic and Content-based C...,Проект предполагает разработку системной комму...,Экосистема «Вулканариум–Ойкумена» объединяет н...
1185,Актив Центра развития карьеры - профориентацио...,Active Club of the Career Development Center -...,Актив Центра развития карьеры — это проектная ...,Актуальность проекта «Актив Центра развития ка...
1186,Обзор Методологии в Области Регионоведения. Эт...,Area Studies Methodology Overview. Data Collec...,Данный этап заключается в поиске и сортировке ...,На текущий момент существует разрозненные пред...


In [14]:
sum(df["title_rus"].isna())

0

In [15]:
df = df.fillna("")

In [16]:
df[df.duplicated(subset=["title_rus"])]

,title_rus,title_eng,annotation,description


In [17]:
df.shape

(1187, 4)

In [18]:
df.drop_duplicates(inplace=True)

In [19]:
df.shape

(1187, 4)

In [20]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can extract short and concise tags for student projects title in russian.
Please, for a given student project title return a list of corresponding tags, which depicts the field of science to which the project is relevant.
For example, for a project title "Рекомендательная система для студенческих проектов" you answer might be the following list of strings with underscores.
["machine_learning", "recommender_systems", "software_engineering"]
Also another example, for a project title "Прогнозирование эффектов процентной политики Банка России" you answer might be the following list of strings with underscores.
["economics", "macroeconomics", "monetary_policy", "forecasting", "time_series"]
Return strictly a list of strings as an answer.
"""

In [21]:
messages = []

In [22]:
for row in df.iterrows():
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[1]["title_rus"]},
        )
    )

In [23]:
tags_set = set()

In [24]:
titles_with_tags_dict = dict()

In [25]:
for message in tqdm.tqdm(messages):
    text = processor.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    answer = json.loads(processor.parse_response(response)["content"])

    titles_with_tags_dict[message[1]["content"]] = answer
    tags_set = tags_set.union(answer)

100%|██████████| 1187/1187 [48:34<00:00,  2.46s/it]


In [26]:
with open("titles_with_tags_dict.pkl", "wb") as f:
    pickle.dump(titles_with_tags_dict, f)

In [27]:
with open("tags_set.pkl", "wb") as f:
    pickle.dump(tags_set, f)

In [28]:
len(tags_set)

1905

In [29]:
list(titles_with_tags_dict.values())[0]

['international_relations',
 'political_economics',
 'policy_analysis',
 'geopolitics',
 'BRICS',
 'international_development']

In [30]:
list(titles_with_tags_dict.values())[-1]

['geography', 'methodology', 'data_collection', 'regional_studies']